# Training Notebook
To be run using on the Google Colab servers

### Global Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision
import numpy as np
import matplotlib.pyplot as plt

# mount google drive
from google.colab import drive
drive.mount('/content/drive')

### Connect to Github repository and import code

In [ ]:
# RUN THIS ONCE AT THE START OF THE SESSION
# authenticate and clone github repository
from getpass import getpass
token = getpass('Enter your GitHub Personal Access Token (PAT): ')
!git clone https://$token@github.com/patrickmastorga/NFL-Big-Data-Bowl-2026.git

# add repo to sys path to allow imports
import sys
sys.path.append('/content/NFL-Big-Data-Bowl-2026')

In [ ]:
# RUN THIS AT THE START AS WELL AS EVERY TIME THE REPO IS UPDATED
# pull changes from github
%cd /content/NFL-Big-Data-Bowl-2026
!git pull https://$token@github.com/patrickmastorga/NFL-Big-Data-Bowl-2026.git

# refrsh the imports
from importlib import reload
import data.dataset
reload(data.dataset)
import models.v1
reload(models.v1)

from data.dataset import BigDataBowlDataset, collate_plays
from models.v1 import Encoder, BallLandingMLP, Decoder, BigDataBowlModelV1

### Load Data

In [ ]:
DATASET_PATH = '/content/NFL-Big-Data-Bowl-2026/data/preprocessed_data.pkl'

# load preprocessed Big Data Bowl Dataset
bdb_preprocessed = BigDataBowlDataset(DATASET_PATH)

### Train Model

In [ ]:
from time import strftime

EPOCHS = 10
BATCH_SIZE = 64
LR = 2e-4

LOG_INTERVAL = 100
MODEL_SAVE_PATH = None # str | None
SAVE_INTERVAL = 1000
LOAD_FROM_CHECKPOINT = None # str | None

# initialize model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder, ball_landing_mlp, decoder = Encoder(), Decoder(), BallLandingMLP()
model = BigDataBowlModelV1(encoder, decoder, ball_landing_mlp).to(device)

# initalize training objects
dataloader = torch.utils.data.DataLoader(
    bdb_preprocessed,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_plays,
    pin_memory=True
)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# training loop
if LOAD_FROM_CHECKPOINT is None:
    train_state_dict = {
        'losses': [],
        'running_loss': 0.0,
        'steps': 0
    }
else:
    train_state_dict = torch.load(LOAD_FROM_CHECKPOINT, map_location=device)
    model.load_state_dict(train_state_dict['model_state'])
    optimizer.load_state_dict(train_state_dict['optimizer_state'])
    print(f'Checkpoint loaded. Resuming from training step {train_state_dict['steps']}.')
total_steps = train_state_dict['steps'] + len(dataloader) * EPOCHS

# training loop
model.train()
print(f'{strftime('%H:%M:%S')} Begin Training')
for epoch in range(EPOCHS):
    for batch_idx, batch in enumerate(dataloader):
        pre_xy =              batch["pre_xy"].to(device, non_blocking=True)
        traj_S =              batch["traj_S"].to(device, non_blocking=True)
        time_remain =         batch["time_remain"].to(device, non_blocking=True)
        roles =               batch["roles"].to(device, non_blocking=True)
        ball_landing =        batch["ball_landing"].to(device, non_blocking=True)
        pre_xy_padding_mask = batch["pre_xy_padding_mask"].to(device, non_blocking=True)
        traj_S_padding_mask = batch["traj_S_padding_mask"].to(device, non_blocking=True)
        roles_padding_mask =  batch["roles_padding_mask"].to(device, non_blocking=True)

        B, A, T_pre, _ = pre_xy.shape
        _, P, T_total, _ = traj_S.shape

        prediction = model(
            pre_xy,
            traj_S,
            time_remain,
            roles,
            ball_landing,
            pre_xy_padding_mask,
            traj_S_padding_mask,
            roles_padding_mask
        ) # (B, P, T_total, D)

        # compute loss
        not_pre_xy = torch.cat([pre_xy_padding_mask[:, :P, :], torch.ones((B, P, T_total - T_pre), dtype=torch.bool, device=device)], dim=2)
        is_predict = ~traj_S_padding_mask and not_pre_xy                        # (B, P, T_total)
        is_predict = is_predict[:, :, 1:, :].unsqueeze(3).expand(-1, -1, -1, 2) # (B, P, T_total - 1, 2)
        diffs = (traj_S[:, :, 1:, :] - prediction[:, :, :-1, :]) * is_predict  # (B, P, T_total - 1, 2)
        loss = (diffs ** 2).sum() / is_predict.sum() # MSE loss

        # gradient descent step
        model.zero_grad()
        loss.backward()
        optimizer.step()

        train_state_dict['running_loss'] += loss.item()
        train_state_dict['steps'] += 1

        # logging step
        if train_state_dict['steps'] % LOG_INTERVAL == 0:
            avg_loss = train_state_dict['running_loss'] / LOG_INTERVAL
            train_state_dict['running_loss'] = 0.0
            train_state_dict['losses'].append((train_state_dict['steps'], avg_loss))
            print(f'{strftime('%H:%M:%S')} TRAINING Step [{train_state_dict['steps']}/{total_steps}]; Loss: {avg_loss:.4f}')

        # saving step
        if MODEL_SAVE_PATH is not None and (train_state_dict['steps'] % SAVE_INTERVAL == 0 or (epoch == EPOCHS - 1 and batch_idx == len(dataloader) - 1)):
            train_state_dict['model_state'] = model.state_dict()
            train_state_dict['optmiizer_state'] = optimizer.state_dict()
            torch.save(train_state_dict, MODEL_SAVE_PATH)
            print(f'{strftime('%H:%M:%S')} Checkpoint saved at step {train_state_dict['steps']} to {MODEL_SAVE_PATH}')
print(f'{strftime('%H:%M:%S')} Training Complete')

# plot training loss
if len(train_state_dict['losses']) > 1:
    plt.figure(figsize=(7, 4))
    plt.plot(*zip(*train_state_dict['losses']))
    plt.grid(True)
    plt.xlabel('Training Step')
    plt.ylabel('Mean Square Error (MSE)')
    plt.title('Training Loss')
    plt.show()